# Week 7 in-class lab — Gradient boosting

This is a contained classroom lab. It is **not** part of Save or Let Go or BookingGuard. Complete the manual update, compare tree methods under one evidence design, then study boosting parameters.

## 1. One residual correction by hand

For squared error, the negative gradient equals the residual. Compute `new_prediction = current_prediction + learning_rate × tree_correction` with learning rate 0.10.

In [ ]:
import numpy as np
import pandas as pd
manual = pd.DataFrame({
    'row':['A','B','C'], 'y':[1.0,0.0,1.0],
    'current_prediction':[0.40,0.30,0.75],
    'tree_correction':[0.50,-0.20,0.20]
})
manual['residual'] = manual.y - manual.current_prediction
manual['new_prediction'] = np.nan  # fill this before revealing the next cell
manual

In [ ]:
LEARNING_RATE = 0.10
answer = manual.copy()
answer['new_prediction'] = answer.current_prediction + LEARNING_RATE * answer.tree_correction
answer['absolute_error_before'] = abs(answer.y - answer.current_prediction)
answer['absolute_error_after'] = abs(answer.y - answer.new_prediction)
answer

**Write:** Why does the learning rate change the correction size without changing its direction? When could a fitted tree correction increase an individual row's error even while total loss falls?

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4))
x = np.arange(len(answer))
ax.scatter(x, answer.y, label='Observed', s=80, color='#172033')
ax.scatter(x, answer.current_prediction, label='Before', s=70, color='#B7791F')
ax.scatter(x, answer.new_prediction, label='After one small correction', s=70, color='#0A8F8F')
for i,r in answer.iterrows():
    ax.plot([i,i],[r.current_prediction,r.new_prediction], color='#2458A6', linewidth=2)
ax.set_xticks(x, answer.row); ax.set_ylabel('Prediction'); ax.legend(); ax.set_title('One additive boosting update');
plt.show()

## 2. Realistic simulated retention case

A subscription business scores renewal risk each week. The generated data include nonlinear effects, interactions, and later-period drift. The final period remains untouched during model selection.

In [ ]:
from sklearn.metrics import log_loss, average_precision_score, roc_auc_score, brier_score_loss
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
rng = np.random.default_rng(42)
n = 3000
week = np.arange(n)
data = pd.DataFrame({
    'week': week,
    'tenure_months': rng.gamma(3, 8, n),
    'monthly_price': rng.normal(75, 18, n).clip(20, 160),
    'support_contacts': rng.poisson(1.3, n),
    'usage_change': rng.normal(0, 0.25, n),
    'annual_plan': rng.integers(0, 2, n),
    'payment_failures': rng.poisson(0.25, n)
})
score = (-1.4 + 0.035*(data.monthly_price-75) - 2.2*data.usage_change + 0.35*data.support_contacts + 0.8*data.payment_failures - 0.7*data.annual_plan + 0.55*((data.monthly_price>90)&(data.tenure_months<12)) + 0.00025*week)
probability = 1/(1+np.exp(-score))
data['cancel'] = rng.binomial(1, probability)
features = [c for c in data if c not in {'cancel','week'}]
train = data.iloc[:2100]; valid = data.iloc[2100:2550]; final_holdout = data.iloc[2550:]
X_train,y_train=train[features],train.cancel
X_valid,y_valid=valid[features],valid.cancel
print({'train_rows':len(train),'validation_rows':len(valid),'holdout_rows_untouched':len(final_holdout)})

In [ ]:
def metrics(name, model):
    model.fit(X_train,y_train)
    p=model.predict_proba(X_valid)[:,1]
    return model,p,{'model':name,'log_loss':log_loss(y_valid,p),'pr_auc':average_precision_score(y_valid,p),'roc_auc':roc_auc_score(y_valid,p),'brier':brier_score_loss(y_valid,p)}
models = {
 'Decision tree': DecisionTreeClassifier(max_depth=4,min_samples_leaf=30,random_state=42),
 'Random Forest': RandomForestClassifier(n_estimators=300,min_samples_leaf=20,max_features='sqrt',n_jobs=1,random_state=42),
 'Histogram boosting': HistGradientBoostingClassifier(learning_rate=.05,max_iter=400,max_leaf_nodes=15,min_samples_leaf=30,l2_regularization=1,early_stopping=True,n_iter_no_change=30,random_state=42)
}
records=[]; fitted={}; probabilities={}
for name,model in models.items():
    fitted[name],probabilities[name],record=metrics(name,model); records.append(record)
pd.DataFrame(records).sort_values('log_loss')

## 3. Boosting parameter laboratory

Change one capacity choice at a time. Explain each run before fitting it. Learning rate and maximum rounds form one decision. Leaf count controls interaction complexity. Minimum leaf support limits fragile local rules.

In [ ]:
grid = [
 {'run':'B0','learning_rate':.05,'max_iter':400,'max_leaf_nodes':15,'min_samples_leaf':30,'hypothesis':'reference'},
 {'run':'B1','learning_rate':.02,'max_iter':900,'max_leaf_nodes':15,'min_samples_leaf':30,'hypothesis':'smaller steps may improve validation'},
 {'run':'B2','learning_rate':.05,'max_iter':400,'max_leaf_nodes':31,'min_samples_leaf':30,'hypothesis':'more interactions may help'},
 {'run':'B3','learning_rate':.05,'max_iter':400,'max_leaf_nodes':15,'min_samples_leaf':10,'hypothesis':'smaller leaves may overfit'}
]
boost_records=[]
for g in grid:
    model=HistGradientBoostingClassifier(learning_rate=g['learning_rate'],max_iter=g['max_iter'],max_leaf_nodes=g['max_leaf_nodes'],min_samples_leaf=g['min_samples_leaf'],l2_regularization=1,early_stopping=True,n_iter_no_change=30,random_state=42)
    model,p,r=metrics(g['run'],model)
    r.update({k:v for k,v in g.items() if k!='run'}); r['best_round']=model.n_iter_; boost_records.append(r)
pd.DataFrame(boost_records).sort_values('log_loss')

**Interpretation:** Which run produced a material and stable improvement? Which parameter increased flexibility? What evidence suggests underfitting or overfitting?

## 4. XGBoost and LightGBM setup

XGBoost adds explicit regularization and optimized tree-building choices. LightGBM uses histogram split search and leaf-wise growth for efficient tabular learning. If a package is unavailable, run `%pip install -q xgboost lightgbm`, restart the runtime, then continue.

In [ ]:
import importlib.util
print({'xgboost_available':importlib.util.find_spec('xgboost') is not None,'lightgbm_available':importlib.util.find_spec('lightgbm') is not None})

In [ ]:
if importlib.util.find_spec('xgboost'):
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=800,learning_rate=.05,max_depth=3,min_child_weight=20,subsample=.8,colsample_bytree=.8,reg_lambda=1,objective='binary:logistic',eval_metric='logloss',random_state=42,n_jobs=1)
    xgb,p,r=metrics('XGBoost',xgb); records.append(r)
else:
    print('Install xgboost to run this comparison.')

In [ ]:
if importlib.util.find_spec('lightgbm'):
    from lightgbm import LGBMClassifier
    lgbm = LGBMClassifier(n_estimators=800,learning_rate=.05,num_leaves=31,min_child_samples=30,subsample=.8,colsample_bytree=.8,reg_lambda=1,verbosity=-1,random_state=42,n_jobs=1)
    lgbm,p,r=metrics('LightGBM',lgbm); records.append(r)
else:
    print('Install lightgbm to run this comparison.')
pd.DataFrame(records).sort_values('log_loss')

## 5. Ideal model selection

Choose the simplest model whose later-period gain is material and stable. Use log loss as the primary probability metric, then check PR-AUC, Brier score, runtime, calibration, and a business guardrail. Do not use the final holdout until the configuration and decision rule are frozen.

In [ ]:
selection = pd.DataFrame(records).sort_values('log_loss').copy()
selection['material_vs_tree'] = selection.log_loss.min() < selection.loc[selection.model=='Decision tree','log_loss'].iloc[0] - .005
selection['decision'] = ''
selection

### Four-line recommendation

1. Model and exact configuration.
2. Two pieces of validation evidence.
3. One failure, uncertainty, or business risk.
4. The next check required before deployment.

### Exit questions

- Why can training loss fall while validation loss rises?
- Which choices link learning rate, rounds, and tree complexity?
- What makes the tree-method comparison fair?
- Which result could block deployment even when aggregate log loss improves?